In [1]:
import rasterio
from rasterio.windows import Window
from pathlib import Path
from pyproj import Transformer
import pandas as pd
import geopandas as gpd
from tqdm.auto import tqdm
from rasterio.crs import CRS
from dist_s1.aws import rasterio_anon_s3_env


In [2]:
PROVISIONAL_PRODUCT_PREFIX = 's3://opera-pst-rs-pop1/products/DIST_S1'

In [3]:
with open('products_in_s3.txt') as f:
    products = list([line.strip() for line in f.readlines()])
    s3_uris_sds = [f'{PROVISIONAL_PRODUCT_PREFIX}/{p}' for p in products]
s3_uris_sds[:2]

['s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T10TDT_20240101T021044Z_20260410T204546Z_S1A_30_v0.1/',
 's3://opera-pst-rs-pop1/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T10TDT_20240101T021044Z_20260421T080831Z_S1A_30_v0.1/']

In [4]:
df = pd.DataFrame({'s3_uri': s3_uris_sds, 'opera_id': products})
df['mgrs_tile_id'] = df['opera_id'].map(lambda x: x.split('_')[3][1:])
df = df.sort_values(by='opera_id').reset_index(drop=True)
df.head()

,s3_uri,opera_id,mgrs_tile_id
0,s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_...,OPERA_L3_DIST-ALERT-S1_T10TDT_20240101T021044Z...,10TDT
1,s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_...,OPERA_L3_DIST-ALERT-S1_T10TDT_20240101T021044Z...,10TDT
2,s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_...,OPERA_L3_DIST-ALERT-S1_T10TDT_20240108T020233Z...,10TDT
3,s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_...,OPERA_L3_DIST-ALERT-S1_T10TDT_20240108T020233Z...,10TDT
4,s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_...,OPERA_L3_DIST-ALERT-S1_T10TDT_20240111T142204Z...,10TDT


In [6]:
mgrst_tile_ids = df.mgrs_tile_id.unique().tolist()
mgrst_tile_ids[:2]

['10TDT', '11TLH']

In [39]:
MGRS_TILE_ID = '37VDH'

In [40]:
import rasterio
from rasterio.windows import Window
from pyproj import Transformer

@rasterio_anon_s3_env
def get_pixel_value(tif, lon, lat):
    with rasterio.open(tif) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        
        x_utm, y_utm = transformer.transform(lon, lat)
        row, col = src.index(x_utm, y_utm)
        if row < 0 or row >= src.height or col < 0 or col >= src.width:
            raise ValueError("Point is outside the bounds of the raster")
        
        # Create window and read pixel value
        window = Window(col, row, 1, 1)
        val = src.read(1, window=window)[0, 0]
    
    return val

In [41]:
mgrs_ts_prods = df[df.mgrs_tile_id == MGRS_TILE_ID].s3_uri.tolist()
mgrs_ts_prods[:2]


['s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T37VDH_20240108T151618Z_20260410T205029Z_S1A_30_v0.1/',
 's3://opera-pst-rs-pop1/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T37VDH_20240120T151618Z_20260410T232058Z_S1A_30_v0.1/']

In [42]:
mgrs_ts_status = [ f'{prod_dir}{Path(prod_dir).name}_GEN-DIST-STATUS.tif' for prod_dir in mgrs_ts_prods]
mgrs_ts_status[:2]

['s3://opera-pst-rs-pop1/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T37VDH_20240108T151618Z_20260410T205029Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T37VDH_20240108T151618Z_20260410T205029Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
 's3://opera-pst-rs-pop1/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T37VDH_20240120T151618Z_20260410T232058Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T37VDH_20240120T151618Z_20260410T232058Z_S1A_30_v0.1_GEN-DIST-STATUS.tif']

In [43]:
mgrs_ts_dates = [pd.to_datetime(Path(prod_dir).name.split('_')[4]).date() for prod_dir in mgrs_ts_prods]
mgrs_ts_dates[:2]

[datetime.date(2024, 1, 8), datetime.date(2024, 1, 20)]

# Dist Tables

In [44]:
df_val = pd.read_csv('../tables/reference_data/selectedpointsLL.csv')
df_val

,ID,Block,subID,blockStratum,substratum,zone,x,y,centx,centy,long,lat,MGRS
0,30961_1,30961,1,treelossTF,3,33,372932.570128,5.453950e+05,372945,545385,13.854048,4.933158,33NUF
1,30961_2,30961,2,treelossTF,2,33,372728.616472,5.425860e+05,372735,542595,13.852197,4.907920,33NUF
2,30961_3,30961,3,treelossTF,1,33,373846.916077,5.470932e+05,373845,547095,13.862138,4.948639,33NUF
3,30961_4,30961,4,treelossTF,4,33,371390.914129,5.460691e+05,371385,546075,13.839969,4.939375,33NUF
4,30961_5,30961,5,treelossTF,3,33,374504.487544,5.464576e+05,374505,546465,13.868099,4.942951,33NUF
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1828,1473302_16,1473302,16,oldcrop_short,2,21,375509.165457,-4.203209e+06,375495,-4203195,-58.417441,-37.967845,21HUU
1829,1473302_17,1473302,17,oldcrop_short,2,21,378407.482557,-4.204245e+06,378405,-4204245,-58.384499,-37.977701,21HUU
1830,1473302_18,1473302,18,oldcrop_short,1,21,374961.521864,-4.197591e+06,374955,-4197585,-58.422612,-37.917224,21HUU
1831,1473302_19,1473302,19,oldcrop_short,4,21,380894.210195,-4.196291e+06,380895,-4196295,-58.354839,-37.906398,21HUU


In [45]:
df_val_mgrs = df_val[df_val['MGRS'] == MGRS_TILE_ID].reset_index(drop=True)
sites_in_mgrs_tile = df_val_mgrs.to_dict(orient='records')
sites_in_mgrs_tile[:2]

[{'ID': '596625_1',
  'Block': 596625,
  'subID': 1,
  'blockStratum': 'wetshort',
  'substratum': 3,
  'zone': 37,
  'x': 487421.58760619,
  'y': 6750102.38764739,
  'centx': 487425,
  'centy': 6750105,
  'long': 38.7683291069216,
  'lat': 60.8859386647085,
  'MGRS': '37VDH'},
 {'ID': '596625_2',
  'Block': 596625,
  'subID': 2,
  'blockStratum': 'wetshort',
  'substratum': 3,
  'zone': 37,
  'x': 487503.155481628,
  'y': 6750210.97091822,
  'centx': 487515,
  'centy': 6750225,
  'long': 38.769979419051,
  'lat': 60.8870188672908,
  'MGRS': '37VDH'}]

In [ ]:
tables = []
for site in tqdm(sites_in_mgrs_tile[:], desc='Processing sites'):
    site_id = site['ID']
    site_lon = site['long']
    site_lat = site['lat'] 

    labels = [get_pixel_value(tif, site_lon, site_lat) for tif in mgrs_ts_status]

    df_dist_s1_table_site = pd.DataFrame({
        'site_id': [site_id] * len(labels),
        'date': mgrs_ts_dates,
        # 'longitude': [site_lon] * len(labels),
        # 'latitude': [site_lat] * len(labels),
        'labels': labels,
    })
    tables.append(df_dist_s1_table_site)

tables[0].head()


Processing sites:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
tables[1].head(20)


,site_id,date,labels
0,824841_2,2024-01-04,0
1,824841_2,2024-01-10,0
2,824841_2,2024-01-16,0
3,824841_2,2024-01-22,0
4,824841_2,2024-01-28,0
5,824841_2,2024-02-03,0
6,824841_2,2024-02-09,0
7,824841_2,2024-02-15,0
8,824841_2,2024-02-21,0
9,824841_2,2024-02-27,0


In [ ]:
for table in tables:
    site_id = table['site_id'].tolist()[0]
    block_id = site_id.split('_')[0]
    table_dir = Path(f'dist_s1_label_tables/{block_id}')
    table_dir.mkdir(exist_ok=True, parents=True)
    table_path = table_dir / f'{site_id}.csv'
    table.to_csv(table_path, index=False)


# Generate Site Vector Table

In [ ]:
df_site_geo = gpd.GeoDataFrame(df_val_mgrs[['ID']],
                               geometry=gpd.points_from_xy(df_val_mgrs.long, 
                                                           df_val_mgrs.lat), 
                               crs=CRS.from_epsg(4326))
df_site_geo.head()


,ID,geometry
0,824841_1,POINT (-89.95 36.10356)
1,824841_2,POINT (-89.97769 36.17506)
2,824841_3,POINT (-89.93527 36.12825)
3,824841_4,POINT (-89.97688 36.1183)
4,824841_5,POINT (-89.9521 36.1327)


In [ ]:
df_site_geo.to_file(table_dir / 'sites.geojson', driver='GeoJSON')